In [ ]:
import h3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download

REPO_ID = "pracadata/brazil-itbi"
FILENAME = "data/belo_horizonte_transactions.parquet"

MAP_YEAR = 2023
START_DATE = "2016-01-01"
END_DATE = "2025-12-31"
RANDOM_STATE = 42

neighbourhoods = ["SAVASSI", "LOURDES", "BURITIS", "CASTELO", "CENTRO"]

In [ ]:
parquet_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=FILENAME,
)

imoveis_df = pd.read_parquet(parquet_path)
imoveis_df["transaction_date"] = pd.to_datetime(
    imoveis_df["transaction_date"]
)

imoveis_df.sample(n=5, random_state=RANDOM_STATE)

,transaction_id,city,transaction_date,date_precision,year,year_month,transaction_type,transaction_type_source,value_declared_brl,value_tax_base_brl,...,neighbourhood,district_code,district_name,postcode5,lat,lon,geo_precision,source_file_sha256,source_url,licence
456183,512bc1234b724446256621a76c6b03a032ebb1e1e1aeef...,belo_horizonte,2024-08-19,day,2024,2024-08,None,None,380000.00,380000.00,...,BURITIS,310620005,Belo Horizonte,30493,-19.974713,-43.965693,face_centroid,9988c2d06cb66ae62cb036bf575f3baa046e69a1eb62c7...,https://ckan.pbh.gov.br/dataset/0e13bf71-5355-...,CC-BY
218233,0d31049e3fa49ff18353c6d4a79e4cc8851730051d8b9b...,belo_horizonte,2015-08-10,day,2015,2015-08,None,None,410000.00,424748.50,...,BURITIS,310620005,Belo Horizonte,30575,-19.979923,-43.974173,face_centroid,4a16a21022d4fae9c801e9048076f69b9830fde95f7775...,https://ckan.pbh.gov.br/dataset/0e13bf71-5355-...,CC-BY
396174,84303e029a11385d834c6637044e0fc82a70fe400c0c56...,belo_horizonte,2022-05-09,day,2022,2022-05,None,None,520068.79,520068.79,...,AEROPORTO,310620005,Belo Horizonte,31270,-19.853800,-43.952690,face_centroid,4a16a21022d4fae9c801e9048076f69b9830fde95f7775...,https://ckan.pbh.gov.br/dataset/0e13bf71-5355-...,CC-BY
502460,98f2b1eb1813a6b9387a10982970597bd897bb8c47f320...,belo_horizonte,2026-04-09,day,2026,2026-04,None,None,200000.00,200000.00,...,VALE DO JATOBA,310620025,Barreiro,30664,-20.007030,-44.039713,face_centroid,928d3d9dbac499b782c9025511722801a06a61af649071...,https://ckan.pbh.gov.br/dataset/0e13bf71-5355-...,CC-BY
293886,0e3162304775ea362e3d4d0cc4eb29c9d02643b9003a3e...,belo_horizonte,2018-06-12,day,2018,2018-06,None,None,25545.60,362094.00,...,CONCORDIA,310620005,Belo Horizonte,31110,-19.898918,-43.934263,postcode_centroid,4a16a21022d4fae9c801e9048076f69b9830fde95f7775...,https://ckan.pbh.gov.br/dataset/0e13bf71-5355-...,CC-BY


In [ ]:
imoveis_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 516725 entries, 0 to 516724
Data columns (total 36 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   transaction_id           516725 non-null  object        
 1   city                     516725 non-null  object        
 2   transaction_date         516725 non-null  datetime64[ns]
 3   date_precision           516725 non-null  object        
 4   year                     516725 non-null  int64         
 5   year_month               516725 non-null  object        
 6   transaction_type         0 non-null       object        
 7   transaction_type_source  0 non-null       object        
 8   value_declared_brl       516725 non-null  object        
 9   value_tax_base_brl       516725 non-null  object        
 10  value_type               516725 non-null  object        
 11  value_flag               15307 non-null   object        
 12  built_area_m2   

In [ ]:
coverage = (
    imoveis_df["geo_precision"]
    .value_counts(dropna=False)
    .rename_axis("precision")
    .reset_index(name="transactions")
)

coverage["percentage"] = (
    100 * coverage["transactions"] / len(imoveis_df)
)

display(coverage)

,precision,transactions,percentage
0,face_centroid,380558,73.648072
1,postcode_centroid,105664,20.448788
2,neighbourhood_centroid,30384,5.880110
3,none,119,0.023030


In [ ]:
map_base = imoveis_df.loc[
    (imoveis_df["year"] == MAP_YEAR)
    & imoveis_df["property_type"].isin(["apartment", "house"])
    & (imoveis_df["geo_precision"] == "face_centroid")
    & imoveis_df["lat"].notna()
    & imoveis_df["lon"].notna()
].copy()

numeric_columns = [
    "lat",
    "lon",
    "built_area_m2",
    "value_declared_brl",
    "price_per_m2_brl",
]

for column in numeric_columns:
    map_base[column] = pd.to_numeric(map_base[column], errors="coerce")

# Cada mapa mantém os próprios critérios e tamanhos originais
type_candidates = map_base.loc[
    (map_base["built_area_m2"] > 0)
    & (map_base["value_declared_brl"] > 0)
]
map_types = type_candidates.sample(
    n=min(6000, len(type_candidates)), random_state=RANDOM_STATE
).copy()

map_types["property_type"] = map_types["property_type"].replace(
    {"apartment": "Apartamento", "house": "Casa"}
)

price_candidates = map_base.loc[
    (map_base["built_area_m2"] > 0)
    & (map_base["price_per_m2_brl"] > 0)
].copy()
price_limits = price_candidates["price_per_m2_brl"].quantile([0.02, 0.98])
price_candidates = price_candidates.loc[
    price_candidates["price_per_m2_brl"].between(*price_limits)
]
map_prices = price_candidates.sample(
    n=min(8000, len(price_candidates)), random_state=RANDOM_STATE
)

value_candidates = map_base.loc[
    (map_base["built_area_m2"] > 0)
    & (map_base["value_declared_brl"] > 0)
].copy()
value_limits = value_candidates["value_declared_brl"].quantile([0.02, 0.98])
value_candidates = value_candidates.loc[
    value_candidates["value_declared_brl"].between(*value_limits)
]
map_values = value_candidates.sample(
    n=min(8000, len(value_candidates)), random_state=RANDOM_STATE
)

print(
    f"Amostras | tipos: {len(map_types):,} | "
    f"R$/m²: {len(map_prices):,} | "
    f"valor declarado: {len(map_values):,}"
)

Amostras | tipos: 6,000 | R$/m²: 8,000 | valor declarado: 8,000


In [ ]:
fig = px.scatter_map(
    map_types,
    lat="lat",
    lon="lon",
    color="property_type",
    size="built_area_m2",
    size_max=14,
    hover_name="neighbourhood",
    hover_data={
        "property_type": True,
        "built_area_m2": ":,.0f",
        "value_declared_brl": ":,.2f",
        "price_per_m2_brl": ":,.2f",
        "building_age_band": True,
        "transaction_date": True,
        "lat": False,
        "lon": False,
    },
    color_discrete_map={
        "Apartamento": "#2563EB",
        "Casa": "#EA580C",
    },
    center={"lat": -19.92, "lon": -43.94},
    zoom=11,
    map_style="carto-positron",
    opacity=0.35,
    height=750,
    title="Mercado imobiliário de Belo Horizonte (2023)",
    labels={
        "property_type": "Tipo de imóvel",
        "built_area_m2": "Área construída (m²)",
        "value_declared_brl": "Valor declarado (R$)",
        "price_per_m2_brl": "Preço por m² (R$)",
        "building_age_band": "Idade da construção",
        "transaction_date": "Data da transação",
    },
)

fig.update_layout(
    title={
        "text": (
            "Belo Horizonte: Geografia do mercado residencial"
            "<br><span style='font-size:13px;color:#64748B'>"
            "Transações de apartamentos e casas (2023)"
            "</span>"
        ),
        "x": 0.025,
        "xanchor": "left",
        "font": {
            "family": "Arial",
            "size": 25,
            "color": "#333333",
        },
    },
    legend={
        "title": {"text": ""},
        "orientation": "h",
        "x": 0.02,
        "xanchor": "left",
        "y": 0.02,
        "yanchor": "bottom",
        "bgcolor": "rgba(255,255,255,0.92)",
        "bordercolor": "rgba(148,163,184,0.35)",
        "borderwidth": 1,
        "font": {"size": 12, "color": "#334155"},
        "itemsizing": "constant",
    },
    margin={"l": 12, "r": 12, "t": 95, "b": 10},
    paper_bgcolor="white",
    font={"family": "Arial", "color": "#334155"},
    height=750,
)

fig.show()

In [ ]:
fig = px.scatter_map(
    map_prices,
    lat="lat",
    lon="lon",
    size="built_area_m2",
    color="price_per_m2_brl",
    color_continuous_scale="Turbo",
    hover_name="neighbourhood",
    hover_data={
        "property_type": True,
        "built_area_m2": ":,.0f",
        "value_declared_brl": ":,.2f",
        "price_per_m2_brl": ":,.2f",
        "transaction_date": True,
        "lat": False,
        "lon": False,
    },
    size_max=16,
    center={"lat": -19.92, "lon": -43.94},
    zoom=11,
    map_style="streets",
    opacity=0.45,
    height=760,
    labels={
        "built_area_m2": "Área construída (m²)",
        "price_per_m2_brl": "Preço por m² (R$)",
        "value_declared_brl": "Valor declarado (R$)",
        "property_type": "Tipo de imóvel",
        "transaction_date": "Data da transação",
    },
    title="Belo Horizonte: Geografia dos valores residenciais"
)

fig.update_layout(
    title={
        "text": (
            "Belo Horizonte: Geografia do mercado residencial"
            "<br><span style='font-size:13px;color:#64748B'>"
            "Tamanho do marcador: área construída - Cor: preço declarado por m² (2023)"
            "</span>"
        ),
        "x": 0.025,
        "xanchor": "left",
        "font": {
            "family": "Arial",
            "size": 25,
            "color": "#333333",
        },
    },
    legend={
        "title": {"text": ""},
        "orientation": "h",
        "x": 0.02,
        "xanchor": "left",
        "y": 0.02,
        "yanchor": "bottom",
        "bgcolor": "rgba(255,255,255,0.92)",
        "bordercolor": "rgba(148,163,184,0.35)",
        "borderwidth": 1,
        "font": {"size": 12, "color": "#334155"},
        "itemsizing": "constant",
    },
    margin={"l": 12, "r": 12, "t": 95, "b": 10},
    paper_bgcolor="white",
    font={"family": "Arial", "color": "#334155"},
    height=750,
)

fig.show()

In [ ]:
fig = px.scatter_map(
    map_values,
    lat="lat",
    lon="lon",
    size="built_area_m2",
    color="value_declared_brl",
    color_continuous_scale="Turbo",
    hover_name="neighbourhood",
    hover_data={
        "property_type": True,
        "built_area_m2": ":,.0f",
        "value_declared_brl": ":,.2f",
        "price_per_m2_brl": ":,.2f",
        "transaction_date": True,
        "lat": False,
        "lon": False,
    },
    size_max=16,
    center={"lat": -19.92, "lon": -43.94},
    zoom=11,
    map_style="streets",
    opacity=0.45,
    height=760,
    labels={
        "built_area_m2": "Área construída (m²)",
        "value_declared_brl": "Valor declarado (R$)",
        "price_per_m2_brl": "Preço por m² (R$)",
    },
    title="Belo Horizonte: Distribuição espacial dos valores declarados"
)

fig.update_layout(
    title={
        "text": (
            "Belo Horizonte: Geografia do mercado"
            "<br><span style='font-size:13px;color:#64748B'>"
            "Tamanho do marcador: área construída - Cor: valor declarado do imóvel (2023)"
            "</span>"
        ),
        "x": 0.025,
        "xanchor": "left",
        "font": {
            "family": "Arial",
            "size": 25,
            "color": "#333333",
        },
    },
    legend={
        "title": {"text": ""},
        "orientation": "h",
        "x": 0.02,
        "xanchor": "left",
        "y": 0.02,
        "yanchor": "bottom",
        "bgcolor": "rgba(255,255,255,0.92)",
        "bordercolor": "rgba(148,163,184,0.35)",
        "borderwidth": 1,
        "font": {"size": 12, "color": "#334155"},
        "itemsizing": "constant",
    },
    margin={"l": 12, "r": 12, "t": 95, "b": 10},
    paper_bgcolor="white",
    font={"family": "Arial", "color": "#334155"},
    height=750,
)

fig.show()

In [ ]:
apartments = imoveis_df.loc[
    (imoveis_df["year"] == MAP_YEAR)
    & (imoveis_df["property_type"] == "apartment")
    & (imoveis_df["geo_precision"] == "face_centroid")
    & imoveis_df["value_flag"].isna()
    & imoveis_df["lat"].notna()
    & imoveis_df["lon"].notna()
    & imoveis_df["price_per_m2_brl"].between(100, 200_000)
].copy()

H3_RESOLUTION = 8
apartments["hex_id"] = [
    h3.latlng_to_cell(lat, lon, H3_RESOLUTION)
    for lat, lon in zip(apartments["lat"], apartments["lon"])
]

hex_stats = (
    apartments.groupby("hex_id", as_index=False)
    .agg(
        median_price_m2=("price_per_m2_brl", "median"),
        n_transactions=("price_per_m2_brl", "size"),
    )
)
hex_stats = hex_stats.loc[hex_stats["n_transactions"] >= 5].copy()

print(f"Eligible transactions: {len(apartments):,}")
print(f"Hexagons with at least 5 transactions: {len(hex_stats):,}")
display(hex_stats.head())

Eligible transactions: 13,997
Hexagons with at least 5 transactions: 255


,hex_id,median_price_m2,n_transactions
2,88a8813683fffff,1958.86,5
3,88a881368bfffff,2557.11,11
6,88a88136c7fffff,3028.11,27
8,88a88136cbfffff,3372.68,7
13,88a88136e9fffff,2065.3,11


In [ ]:
series_data = imoveis_df.loc[
    imoveis_df["transaction_date"].between(START_DATE, END_DATE)
    & imoveis_df["property_type"].isin(["apartment", "house"])
    & imoveis_df["value_flag"].isna()
    & imoveis_df["price_per_m2_brl"].between(100, 200_000)
    & imoveis_df["price_per_m2_brl_real"].notna()
].copy()

series_data["month"] = series_data["transaction_date"].dt.to_period("M").dt.to_timestamp()
series_data["quarter"] = series_data["transaction_date"].dt.to_period("Q").dt.start_time
series_data["neighbourhood"] = (
    series_data["neighbourhood"].astype("string").str.strip().str.upper()
)

# Convert to float to avoid Decimal vs Float multiplication errors in quantiles
series_data["price_per_m2_brl_real"] = series_data["price_per_m2_brl_real"].astype(float)

monthly_prices = (
    series_data.groupby(["month", "property_type"])
    .agg(
        n_transactions=("price_per_m2_brl_real", "size"),
        p25=("price_per_m2_brl_real", lambda values: values.quantile(0.25)),
        median=("price_per_m2_brl_real", "median"),
        p75=("price_per_m2_brl_real", lambda values: values.quantile(0.75)),
    )
    .reset_index()
)
monthly_prices = monthly_prices.loc[monthly_prices["n_transactions"] >= 30]

# Reaproveita os mesmos quantis para o gráfico de apartamentos, sem
# agrupar e calcular os quantis uma segunda vez
monthly_apartment_prices = (
    monthly_prices.loc[
        monthly_prices["property_type"] == "apartment",
        ["month", "p25", "median", "p75", "n_transactions"],
    ]
    .rename(columns={"n_transactions": "transactions"})
    .copy()
)

print(f"Registros elegíveis (2016–2025): {len(series_data):,}")
display(monthly_prices.head())


Registros elegíveis (2016–2025): 205,436


,month,property_type,n_transactions,p25,median,p75
0,2016-01-01,apartment,1410,4090.5725,5084.055,6109.6300
1,2016-01-01,house,188,2251.9550,3854.755,6182.2825
2,2016-02-01,apartment,1458,4026.0650,5021.380,6022.7875
3,2016-02-01,house,191,1591.3600,3681.620,5390.3300
4,2016-03-01,apartment,1632,3978.3400,4983.485,5927.8725


In [ ]:
styles = {
    "apartment": {
        "label": "Apartamentos",
        "line": "#2563EB",
        "fill": "rgba(37, 99, 235, 0.12)",
    },
    "house": {
        "label": "Casas",
        "line": "#EA580C",
        "fill": "rgba(234, 88, 12, 0.12)",
    },
}

fig = go.Figure()

for property_type, style in styles.items():
    subset = (
        monthly_prices.loc[
            monthly_prices["property_type"] == property_type
        ]
        .sort_values("month")
    )

    # Upper boundary of the interquartile ribbon
    fig.add_trace(
        go.Scatter(
            x=subset["month"],
            y=subset["p75"],
            mode="lines",
            line={"width": 0},
            showlegend=False,
            hoverinfo="skip",
            legendgroup=property_type,
        )
    )

    # Lower boundary, filled towards the upper boundary
    fig.add_trace(
        go.Scatter(
            x=subset["month"],
            y=subset["p25"],
            mode="lines",
            fill="tonexty",
            fillcolor=style["fill"],
            line={"width": 0},
            showlegend=False,
            hoverinfo="skip",
            legendgroup=property_type,
        )
    )

    # Mediana
    fig.add_trace(
        go.Scatter(
            x=subset["month"],
            y=subset["median"],
            mode="lines",
            name=style["label"],
            line={
                "color": style["line"],
                "width": 2.7,
            },
            customdata=subset[["n_transactions"]].to_numpy(),
            hovertemplate=(
                "Mediana: R$ %{y:,.0f}/m²"
                "<br>Transações: %{customdata[0]:,}"
                "<extra></extra>"
            ),
            legendgroup=property_type,
        )
    )

fig.update_layout(
    title={
        "text": (
            "Belo Horizonte: Evolução dos valores residenciais"
            "<br><sup>"
            "Valor declarado por m², corrigido pelo IPCA"
            " - Mediana e intervalo interquartil (2016–2025)"
            "</sup>"
        ),
        "x": 0.02,
        "xanchor": "left",
        "font": {"size": 21, "color": "#172033"},
    },
    xaxis={
        "title": None,
        "dtick": "M12",
        "tickformat": "%Y",
        "showgrid": False,
    },
    yaxis={
        "title": "Valor declarado por m² (R$ constantes)",
        "tickprefix": "R$ ",
        "tickformat": ",.0f",
        "gridcolor": "#E2E8F0",
        "zeroline": False,
    },
    legend={
        "title": {"text": ""},
        "orientation": "h",
        "x": 0.02,
        "y": 1.05,
        "xanchor": "left",
    },
    hovermode="x unified",
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=630,
    margin={"l": 85, "r": 30, "t": 115, "b": 65},
    font={"family": "Arial", "color": "#334155"},
)

fig.show()

In [ ]:
fig_distribution = go.Figure()

fig_distribution.add_trace(
    go.Scatter(
        x=monthly_apartment_prices["month"],
        y=monthly_apartment_prices["p25"],
        mode="lines",
        line={"width": 0},
        hoverinfo="skip",
        showlegend=False,
    )
)

fig_distribution.add_trace(
    go.Scatter(
        x=monthly_apartment_prices["month"],
        y=monthly_apartment_prices["p75"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(15,118,110,0.16)",
        line={"width": 0},
        name="Intervalo interquartil (P25–P75)",
        hoverinfo="skip",
    )
)

fig_distribution.add_trace(
    go.Scatter(
        x=monthly_apartment_prices["month"],
        y=monthly_apartment_prices["median"],
        mode="lines",
        name="Mediana",
        line={"color": "#0F766E", "width": 3},
        customdata=monthly_apartment_prices[
            ["p25", "p75", "transactions"]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{x|%b %Y}</b><br>"
            "Mediana: R$ %{y:,.0f}/m²<br>"
            "P25: R$ %{customdata[0]:,.0f}/m²<br>"
            "P75: R$ %{customdata[1]:,.0f}/m²<br>"
            "Observações: %{customdata[2]:,.0f}"
            "<extra></extra>"
        ),
    )
)

fig_distribution.update_layout(
    title={
        "text": (
            "Belo Horizonte: Evolução dos valores residenciais"
            "<br><span style='font-size:13px;color:#64748B'>"
            "Apartamentos - Valor declarado por m², corrigido pelo IPCA"
            " (2016–2025)"
            "</span>"
        ),
        "x": 0.02,
        "xanchor": "left",
        "font": {"size": 21, "color": "#172033"},
    },
    template="plotly_white",
    height=560,
    hovermode="x unified",
    legend={
        "orientation": "h",
        "x": 0,
        "y": 1.04,
        "title": {"text": ""},
    },
    margin={"l": 75, "r": 30, "t": 125, "b": 70},
    font={"family": "Arial", "color": "#334155"},
)

fig_distribution.update_yaxes(
    title_text="Valor declarado real (R$/m²)",
    tickprefix="R$ ",
    tickformat=",.0f",
    gridcolor="#E2E8F0",
)

fig_distribution.update_xaxes(
    title_text="",
    dtick="M12",
    tickformat="%Y",
    showgrid=False,
    rangeslider_visible=True,
)

fig_distribution.show()

In [ ]:
selected_data = series_data.loc[
    (series_data["property_type"] == "apartment")
    & series_data["neighbourhood"].isin(neighbourhoods)
]

quarterly_prices = (
    selected_data.groupby(["neighbourhood", "quarter"])["price_per_m2_brl_real"]
    .agg(median_price="median", transactions="size")
)

quarters = pd.date_range("2016-01-01", "2025-10-01", freq="QS")
complete_index = pd.MultiIndex.from_product(
    [neighbourhoods, quarters],
    names=["neighbourhood", "quarter"],
)
quarterly_prices = quarterly_prices.reindex(complete_index).reset_index()
quarterly_prices.loc[
    quarterly_prices["transactions"].fillna(0) < 15,
    "median_price",
] = float("nan")

display(quarterly_prices.head())

,neighbourhood,quarter,median_price,transactions
0,SAVASSI,2016-01-01,7108.880,95
1,SAVASSI,2016-04-01,6378.290,63
2,SAVASSI,2016-07-01,7314.100,65
3,SAVASSI,2016-10-01,7500.805,84
4,SAVASSI,2017-01-01,6986.190,97


In [ ]:
neighbourhood_colors = {
    "SAVASSI": "#0F766E",
    "LOURDES": "#2563EB",
    "BURITIS": "#D97706",
    "CASTELO": "#7C3AED",
    "CENTRO": "#DB2777",
}

fig_neighbourhoods = px.line(
    quarterly_prices,
    x="quarter",
    y="median_price",
    color="neighbourhood",
    color_discrete_map=neighbourhood_colors,
    category_orders={"neighbourhood": neighbourhoods},
    custom_data=["transactions"],
    markers=True,
    labels={
        "quarter": "Trimestre",
        "median_price": "Valor mediano real (R$/m²)",
        "neighbourhood": "Bairro",
    },
)

fig_neighbourhoods.update_traces(
    line={"width": 2.5},
    marker={"size": 5},
    connectgaps=False,
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Trimestre iniciado em %{x|%b %Y}<br>"
        "Mediana: R$ %{y:,.0f}/m²<br>"
        "Observações: %{customdata[0]:,.0f}"
        "<extra></extra>"
    ),
)

fig_neighbourhoods.update_layout(
    title={
        "text": (
            "Belo Horizonte: Trajetórias dos bairros"
            "<br><span style='font-size:13px;color:#64748B'>"
            "Apartamentos - Mediana trimestral do valor declarado por m²"
            " - Valores reais (2016–2025)"
            "</span>"
        ),
        "x": 0.02,
        "xanchor": "left",
        "font": {"size": 21, "color": "#172033"},
    },
    template="plotly_white",
    height=620,
    hovermode="closest",
    legend={
        "title": {"text": ""},
        "orientation": "h",
        "x": 0,
        "y": 1.04,
    },
    margin={"l": 80, "r": 30, "t": 125, "b": 70},
    font={"family": "Arial", "color": "#334155"},
)

fig_neighbourhoods.update_yaxes(
    title_text="Valor declarado real (R$/m²)",
    tickprefix="R$ ",
    tickformat=",.0f",
    gridcolor="#E2E8F0",
)

fig_neighbourhoods.update_xaxes(
    title_text="",
    dtick="M12",
    tickformat="%Y",
    showgrid=False,
)

fig_neighbourhoods.show()

In [ ]:
import numpy as np
from plotly.subplots import make_subplots

# Decimal columns become floats once, so every figure below can do arithmetic.
NUMERIC_COLUMNS: list[str] = [
    "value_declared_brl",
    "value_tax_base_brl",
    "value_declared_brl_real",
    "built_area_m2",
    "price_per_m2_brl",
    "price_per_m2_brl_real",
]

df = imoveis_df.copy()
df[NUMERIC_COLUMNS] = df[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
df["neighbourhood"] = df["neighbourhood"].astype("string").str.strip().str.upper()

# Same eligibility rules as the time-series cells: no flagged values,
# plausible price per m², apartments and houses only.
residential = df.loc[
    df["property_type"].isin(["apartment", "house"])
    & df["value_flag"].isna()
    & (df["built_area_m2"] > 0)
    & df["price_per_m2_brl"].between(100, 200_000)
].copy()
residential["tipo"] = residential["property_type"].map(
    {"apartment": "Apartamentos", "house": "Casas"}
)

INK: str = "#172033"
MUTED: str = "#64748B"
GRID: str = "#E2E8F0"
BLUE: str = "#2563EB"
ORANGE: str = "#EA580C"
TEAL: str = "#0F766E"
NEUTRAL: str = "#CBD5E1"
TYPE_COLORS: dict[str, str] = {"Apartamentos": BLUE, "Casas": ORANGE}

# One hue, light to dark: magnitude reads as darkness and nothing else.
SEQUENTIAL: list[str] = [
    "#CCFBF1", "#99F6E4", "#5EEAD4", "#2DD4BF",
    "#14B8A6", "#0D9488", "#0F766E", "#115E59", "#134E4A",
]
LEGEND: dict = {
    "title": {"text": ""},
    "orientation": "h",
    "x": 0,
    "y": 1.0,
    "yanchor": "bottom",
}


def make_title(title: str, subtitle: str) -> dict:
    """Return the two-line title used by every figure in this notebook."""
    return {
        "text": (
            f"{title}<br>"
            f"<span style='font-size:13px;color:{MUTED}'>{subtitle}</span>"
        ),
        "x": 0.02,
        "xanchor": "left",
        "font": {"size": 21, "color": INK},
    }


BASE_LAYOUT: dict = {
    "template": "plotly_white",
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "font": {"family": "Arial", "color": "#334155"},
    "margin": {"l": 80, "r": 30, "t": 130, "b": 65},
    "hoverlabel": {"bgcolor": "white", "font": {"family": "Arial"}},
    # Brazilian number formatting: comma for decimals, dot for thousands.
    "separators": ",.",
}

print(f"Transações residenciais elegíveis: {len(residential):,}")

Transações residenciais elegíveis: 398,678


In [ ]:
share_data = residential.loc[residential["value_tax_base_brl"] > 0].copy()
share_data["quarter"] = (
    share_data["transaction_date"].dt.to_period("Q").dt.start_time
)
share_data["below_base"] = (
    share_data["value_declared_brl"] < share_data["value_tax_base_brl"]
)

quarterly_share = (
    share_data.groupby(["quarter", "tipo"])
    .agg(share=("below_base", "mean"), transactions=("below_base", "size"))
    .reset_index()
)
quarterly_share = quarterly_share.loc[quarterly_share["transactions"] >= 100]

fig_share = px.line(
    quarterly_share,
    x="quarter",
    y="share",
    color="tipo",
    color_discrete_map=TYPE_COLORS,
    custom_data=["transactions"],
)

fig_share.update_traces(
    line={"width": 2.5},
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Abaixo da base: %{y:.1%}<br>"
        "Transações: %{customdata[0]:,}"
        "<extra></extra>"
    ),
)

fig_share.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: Declarar abaixo da base ficou menos comum",
        "Parcela trimestral das transações com valor declarado inferior "
        "à base de cálculo do ITBI (2008–2026)",
    ),
    height=560,
    hovermode="x unified",
    legend=LEGEND,
)
fig_share.update_xaxes(
    title_text="", dtick="M24", tickformat="%Y", showgrid=False
)
fig_share.update_yaxes(
    title_text="Transações abaixo da base de cálculo",
    tickformat=".0%",
    range=[0, 1],
    gridcolor=GRID,
    zeroline=False,
)

fig_share.show()

In [ ]:
BIN_WIDTH: int = 5_000
ROUND_STEP: int = 50_000

heaping_data = residential.loc[
    residential["year"].between(2023, 2025)
    & residential["value_declared_brl"].between(50_000, 1_000_000)
]

# Bins are centred on multiples of R$ 5 mil, so a round price sits mid-bin.
bin_centres = (
    (heaping_data["value_declared_brl"] / BIN_WIDTH).round() * BIN_WIDTH
).astype(int)
heaping = (
    bin_centres.value_counts()
    .sort_index()
    .rename_axis("value")
    .reset_index(name="transactions")
)
heaping["is_round"] = heaping["value"] % ROUND_STEP == 0
round_share = (
    heaping.loc[heaping["is_round"], "transactions"].sum()
    / heaping["transactions"].sum()
)

fig_heaping = go.Figure()
for is_round, label, color in [
    (False, "Demais valores", NEUTRAL),
    (True, "Múltiplos de R$ 50 mil", TEAL),
]:
    subset = heaping.loc[heaping["is_round"] == is_round]
    fig_heaping.add_trace(
        go.Bar(
            x=subset["value"],
            y=subset["transactions"],
            width=BIN_WIDTH * 0.9,
            name=label,
            marker={"color": color, "line": {"width": 0}},
            hovertemplate=(
                "R$ %{x:,.0f}<br>Transações: %{y:,}<extra></extra>"
            ),
        )
    )

fig_heaping.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: Preços declarados preferem números redondos",
        # A single "$" per string: two of them would be read as LaTeX.
        f"Apartamentos e casas (2023–2025), faixas de 5 mil reais - "
        f"{round_share:.0%} das transações caem em múltiplos de 50 mil, "
        f"que ocupam cerca de 10% das faixas",
    ),
    height=560,
    barmode="overlay",
    legend=LEGEND,
)
fig_heaping.update_xaxes(
    title_text="Valor declarado",
    tickprefix="R$ ",
    tickformat=",.0f",
    dtick=100_000,
    showgrid=False,
)
fig_heaping.update_yaxes(
    title_text="Transações", tickformat=",.0f", gridcolor=GRID, zeroline=False
)

fig_heaping.show()

In [ ]:
TOP_N: int = 30
PERIODS: dict[str, tuple[int, int]] = {
    "2016–2017": (2016, 2017),
    "2024–2025": (2024, 2025),
}

apartment_sales = residential.loc[
    (residential["property_type"] == "apartment")
    & residential["price_per_m2_brl_real"].notna()
]
top_neighbourhoods = (
    apartment_sales.loc[apartment_sales["year"].between(2016, 2025)]
    ["neighbourhood"].value_counts().head(TOP_N).index
)

# Two-year windows keep each median on a few hundred transactions.
period_medians = pd.DataFrame(
    {
        label: apartment_sales.loc[
            apartment_sales["year"].between(start, end)
            & apartment_sales["neighbourhood"].isin(top_neighbourhoods)
        ]
        .groupby("neighbourhood")["price_per_m2_brl_real"]
        .median()
        for label, (start, end) in PERIODS.items()
    }
)
first_period, last_period = PERIODS
period_medians["change"] = (
    period_medians[last_period] / period_medians[first_period] - 1
)
period_medians = period_medians.sort_values(last_period)
period_medians.index = period_medians.index.str.title()

fig_dumbbell = go.Figure()

for neighbourhood, row in period_medians.iterrows():
    fig_dumbbell.add_trace(
        go.Scatter(
            x=[row[first_period], row[last_period]],
            y=[neighbourhood, neighbourhood],
            mode="lines",
            line={"color": NEUTRAL, "width": 2},
            hoverinfo="skip",
            showlegend=False,
        )
    )

for period, color in [(first_period, "#94A3B8"), (last_period, TEAL)]:
    fig_dumbbell.add_trace(
        go.Scatter(
            x=period_medians[period],
            y=period_medians.index,
            mode="markers",
            name=period,
            marker={
                "color": color,
                "size": 10,
                "line": {"color": "white", "width": 1.5},
            },
            customdata=period_medians[["change"]].to_numpy(),
            hovertemplate=(
                "<b>%{y}</b><br>"
                f"{period}: R$ %{{x:,.0f}}/m²<br>"
                "Variação real no período: %{customdata[0]:+.1%}"
                "<extra></extra>"
            ),
        )
    )

fig_dumbbell.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: Quem subiu e quem caiu em termos reais",
        f"Apartamentos - Mediana do valor declarado por m², corrigido pelo "
        f"IPCA - {TOP_N} bairros com mais transações",
    ),
    height=900,
    legend=LEGEND,
)
fig_dumbbell.update_layout(margin={"l": 150})
fig_dumbbell.update_xaxes(
    title_text="Valor declarado real (R$/m²)",
    tickprefix="R$ ",
    tickformat=",.0f",
    gridcolor=GRID,
    zeroline=False,
)
fig_dumbbell.update_yaxes(title_text="", showgrid=False)

fig_dumbbell.show()

In [ ]:
MONTH_LABELS: list[str] = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
                           "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

calendar = (
    df.assign(month=df["transaction_date"].dt.month)
    .pivot_table(
        index="year", columns="month", values="transaction_id", aggfunc="size"
    )
    .reindex(columns=range(1, 13))
)

# One backlog month would flatten every other cell, so the colour scale
# stops at the 98th percentile; hover still reports the true count.
color_cap = float(np.nanquantile(calendar.to_numpy(), 0.98))

fig_calendar = go.Figure(
    go.Heatmap(
        x=MONTH_LABELS,
        y=calendar.index,
        z=calendar.to_numpy(),
        zmin=0,
        zmax=color_cap,
        colorscale=SEQUENTIAL,
        xgap=2,
        ygap=2,
        hoverongaps=False,
        colorbar={
            "title": {"text": "Transações"},
            "tickformat": ",.0f",
            "thickness": 12,
            "len": 0.6,
            "outlinewidth": 0,
        },
        hovertemplate="%{x} de %{y}<br>Transações: %{z:,}<extra></extra>",
    )
)

fig_calendar.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: O calendário do mercado imobiliário",
        "Transações registradas no ITBI por mês, todos os tipos de imóvel "
        "- Escala de cor limitada ao percentil 98",
    ),
    height=680,
)
fig_calendar.update_xaxes(side="top", showgrid=False)
fig_calendar.update_yaxes(
    title_text="", autorange="reversed", dtick=1, showgrid=False
)

fig_calendar.show()

In [ ]:
AGE_LABELS: dict[str, str] = {
    "new": "Novo",
    "3_10": "3 a 10 anos",
    "11_25": "11 a 25 anos",
    "26_plus": "26 anos ou mais",
}
AGE_COLORS: dict[str, str] = dict(
    zip(AGE_LABELS.values(), ["#134E4A", "#0D9488", "#5EEAD4", "#94A3B8"])
)

age_data = residential.loc[
    (residential["property_type"] == "apartment")
    & residential["year"].between(2023, 2025)
    & residential["building_age_band"].isin(AGE_LABELS)
    & residential["price_per_m2_brl_real"].notna()
].copy()
age_data["idade"] = age_data["building_age_band"].map(AGE_LABELS)
price_cap = age_data["price_per_m2_brl_real"].quantile(0.99)
age_data = age_data.loc[age_data["price_per_m2_brl_real"] <= price_cap]

age_quartiles = (
    age_data.groupby("idade")["price_per_m2_brl_real"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
)

fig_age = go.Figure()
# Oldest band first, so the newest ends up at the top of the axis.
for label, color in reversed(AGE_COLORS.items()):
    subset = age_data.loc[age_data["idade"] == label]
    fig_age.add_trace(
        go.Violin(
            x=subset["price_per_m2_brl_real"],
            y0=label,
            orientation="h",
            side="positive",
            width=1.7,
            points=False,
            spanmode="hard",
            line={"color": color, "width": 1.5},
            fillcolor=color,
            opacity=0.8,
            hoverinfo="skip",
        )
    )
    p25, median, p75 = age_quartiles.loc[label]
    fig_age.add_trace(
        go.Scatter(
            x=[median],
            y=[label],
            mode="markers",
            marker={
                "color": INK,
                "size": 9,
                "line": {"color": "white", "width": 1.5},
            },
            hovertemplate=(
                f"<b>{label}</b><br>"
                f"Mediana: R$ {median:,.0f}/m²<br>"
                f"P25–P75: {p25:,.0f} a {p75:,.0f}<br>"
                f"Transações: {len(subset):,}"
                "<extra></extra>"
            ).replace(",", "."),
        )
    )

fig_age.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: O prêmio do apartamento novo",
        "Distribuição do valor declarado por m², corrigido pelo IPCA, "
        "por idade da construção (2023–2025) - Ponto: mediana - "
        "1% superior omitido",
    ),
    height=600,
    showlegend=False,
    violingap=0,
)
fig_age.update_layout(margin={"l": 130})
fig_age.update_xaxes(
    title_text="Valor declarado real (R$/m²)",
    tickprefix="R$ ",
    tickformat=",.0f",
    gridcolor=GRID,
    zeroline=False,
)
fig_age.update_yaxes(title_text="", showgrid=False)

fig_age.show()

In [ ]:
age_series = residential.loc[
    (residential["property_type"] == "apartment")
    & residential["year"].between(2016, 2025)
    & residential["building_age_band"].isin(AGE_LABELS)
    & residential["price_per_m2_brl_real"].notna()
].copy()
age_series["idade"] = age_series["building_age_band"].map(AGE_LABELS)

yearly_age = (
    age_series.groupby(["year", "idade"])["price_per_m2_brl_real"]
    .agg(median="median", transactions="size")
    .reset_index()
)

fig_age_series = px.line(
    yearly_age,
    x="year",
    y="median",
    color="idade",
    color_discrete_map=AGE_COLORS,
    category_orders={"idade": list(AGE_COLORS)},
    custom_data=["transactions"],
    markers=True,
)

fig_age_series.update_traces(
    line={"width": 2.5},
    marker={"size": 7, "line": {"color": "white", "width": 1.5}},
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Mediana: R$ %{y:,.0f}/m²<br>"
        "Transações: %{customdata[0]:,}"
        "<extra></extra>"
    ),
)

fig_age_series.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: Novos e usados ao longo do tempo",
        "Apartamentos - Mediana anual do valor declarado por m², corrigido "
        "pelo IPCA, por idade da construção (2016–2025)",
    ),
    height=560,
    hovermode="x unified",
    legend=LEGEND,
)
fig_age_series.update_xaxes(title_text="", dtick=1, showgrid=False)
fig_age_series.update_yaxes(
    title_text="Valor declarado real (R$/m²)",
    tickprefix="R$ ",
    tickformat=",.0f",
    gridcolor=GRID,
    zeroline=False,
)

fig_age_series.show()

In [ ]:
PRACA_SETE: tuple[float, float] = (-19.9191, -43.9386)
EARTH_RADIUS_KM: float = 6_371.0
MAX_DISTANCE_KM: int = 14
PERIOD_COLORS: dict[str, str] = {
    "2016–2018": "#94A3B8",
    "2019–2021": "#2DD4BF",
    "2022–2025": "#115E59",
}

gradient_data = residential.loc[
    (residential["property_type"] == "apartment")
    & (residential["geo_precision"] == "face_centroid")
    & residential["year"].between(2016, 2025)
    & residential["price_per_m2_brl_real"].notna()
].copy()

# Haversine distance from each street-face centroid to Praça Sete.
lat = np.radians(gradient_data["lat"])
lon = np.radians(gradient_data["lon"])
centre_lat, centre_lon = np.radians(PRACA_SETE)
haversine = (
    np.sin((lat - centre_lat) / 2) ** 2
    + np.cos(lat) * np.cos(centre_lat) * np.sin((lon - centre_lon) / 2) ** 2
)
gradient_data["distance_km"] = 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(haversine))

gradient_data = gradient_data.loc[gradient_data["distance_km"] < MAX_DISTANCE_KM]
gradient_data["band_km"] = np.floor(gradient_data["distance_km"]) + 0.5
gradient_data["period"] = pd.cut(
    gradient_data["year"],
    bins=[2015, 2018, 2021, 2025],
    labels=list(PERIOD_COLORS),
).astype(str)

gradient = (
    gradient_data.groupby(["period", "band_km"])["price_per_m2_brl_real"]
    .agg(median="median", transactions="size")
    .reset_index()
)
gradient = gradient.loc[gradient["transactions"] >= 50]

fig_gradient = px.line(
    gradient,
    x="band_km",
    y="median",
    color="period",
    color_discrete_map=PERIOD_COLORS,
    category_orders={"period": list(PERIOD_COLORS)},
    custom_data=["transactions"],
    markers=True,
)

fig_gradient.update_traces(
    line={"width": 2.5},
    marker={"size": 7, "line": {"color": "white", "width": 1.5}},
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Mediana: R$ %{y:,.0f}/m²<br>"
        "Transações: %{customdata[0]:,}"
        "<extra></extra>"
    ),
)

fig_gradient.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: O preço em função da distância do centro",
        "Apartamentos - Mediana do valor declarado real por m² em faixas "
        "de 1 km a partir da Praça Sete (mínimo de 50 transações)",
    ),
    height=560,
    hovermode="x unified",
    legend=LEGEND,
)
fig_gradient.update_xaxes(
    title_text="Distância da Praça Sete (km, em linha reta)",
    dtick=1,
    tick0=0,
    range=[0, MAX_DISTANCE_KM],
    showgrid=False,
)
fig_gradient.update_yaxes(
    title_text="Valor declarado real (R$/m²)",
    tickprefix="R$ ",
    tickformat=",.0f",
    gridcolor=GRID,
    zeroline=False,
)

fig_gradient.show()

In [ ]:
import statsmodels.formula.api as smf

hedonic_data = residential.loc[
    residential["year"].between(2016, 2025)
    & residential["price_per_m2_brl_real"].notna()
    & residential["building_age_band"].notna()
    & residential["neighbourhood"].notna()
].copy()
hedonic_data["quarter"] = (
    hedonic_data["transaction_date"].dt.to_period("Q").astype(str)
)
hedonic_data["log_price"] = np.log(hedonic_data["price_per_m2_brl_real"])
hedonic_data["log_area"] = np.log(hedonic_data["built_area_m2"])

# Neighbourhood effects need enough sales to be estimated at all.
neighbourhood_sizes = hedonic_data["neighbourhood"].value_counts()
hedonic_data = hedonic_data.loc[
    hedonic_data["neighbourhood"].isin(
        neighbourhood_sizes.index[neighbourhood_sizes >= 100]
    )
]

# Quarter effects, holding size, age, type and neighbourhood fixed.
# Standard errors are clustered by neighbourhood.
hedonic_model = smf.ols(
    "log_price ~ log_area + C(building_age_band) + C(property_type)"
    " + C(neighbourhood) + C(quarter)",
    data=hedonic_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": hedonic_data["neighbourhood"].astype("category").cat.codes
    },
)

quarter_terms = hedonic_model.params.filter(like="C(quarter)").index
intervals = hedonic_model.conf_int().loc[quarter_terms]
hedonic_index = pd.DataFrame(
    {
        "quarter": quarter_terms.str.extract(r"T\.(\w+)\]")[0].to_numpy(),
        "index": 100 * np.exp(hedonic_model.params[quarter_terms].to_numpy()),
        "lower": 100 * np.exp(intervals[0].to_numpy()),
        "upper": 100 * np.exp(intervals[1].to_numpy()),
    }
)
base_quarter = hedonic_data["quarter"].min()
hedonic_index = pd.concat(
    [
        pd.DataFrame(
            {"quarter": [base_quarter], "index": [100.0],
             "lower": [100.0], "upper": [100.0]}
        ),
        hedonic_index,
    ],
    ignore_index=True,
)
hedonic_index["date"] = pd.PeriodIndex(
    hedonic_index["quarter"], freq="Q"
).start_time

raw_median = hedonic_data.groupby("quarter")["price_per_m2_brl_real"].median()
hedonic_index["raw_index"] = (
    100 * raw_median / raw_median[base_quarter]
).reindex(hedonic_index["quarter"]).to_numpy()

print(
    f"Observações: {int(hedonic_model.nobs):,} | "
    f"R²: {hedonic_model.rsquared:.3f} | "
    f"Elasticidade do preço por m² à área: "
    f"{hedonic_model.params['log_area']:.3f}"
)

fig_hedonic = go.Figure()

fig_hedonic.add_trace(
    go.Scatter(
        x=hedonic_index["date"],
        y=hedonic_index["upper"],
        mode="lines",
        line={"width": 0},
        hoverinfo="skip",
        showlegend=False,
    )
)
fig_hedonic.add_trace(
    go.Scatter(
        x=hedonic_index["date"],
        y=hedonic_index["lower"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(15,118,110,0.16)",
        line={"width": 0},
        hoverinfo="skip",
        showlegend=False,
    )
)
fig_hedonic.add_trace(
    go.Scatter(
        x=hedonic_index["date"],
        y=hedonic_index["raw_index"],
        mode="lines",
        name="Mediana bruta",
        line={"color": "#94A3B8", "width": 2, "dash": "dot"},
        hovertemplate="Mediana bruta: %{y:.1f}<extra></extra>",
    )
)
fig_hedonic.add_trace(
    go.Scatter(
        x=hedonic_index["date"],
        y=hedonic_index["index"],
        mode="lines",
        name="Índice hedônico (IC 95%)",
        line={"color": TEAL, "width": 3},
        hovertemplate="Índice hedônico: %{y:.1f}<extra></extra>",
    )
)

fig_hedonic.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: O preço de um imóvel de qualidade constante",
        "Índice hedônico trimestral do valor declarado real por m², "
        "controlando área, idade, tipo e bairro - Apartamentos e casas",
    ),
    height=580,
    hovermode="x unified",
    legend=LEGEND,
)
fig_hedonic.update_xaxes(
    title_text="", dtick="M12", tickformat="%Y", showgrid=False
)
fig_hedonic.update_yaxes(
    title_text=f"Índice ({base_quarter[-1]}º tri. de {base_quarter[:4]} = 100)",
    gridcolor=GRID,
    zeroline=False,
)

fig_hedonic.show()

Observações: 202,204 | R²: 0.237 | Elasticidade do preço por m² à área: -0.155


In [ ]:
QUANTILES: list[float] = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
EFFECTS: dict[str, str] = {
    "C(building_age_band, Treatment('26_plus'))[T.new]": "Novo",
    "C(building_age_band, Treatment('26_plus'))[T.3_10]": "3 a 10 anos",
    "C(building_age_band, Treatment('26_plus'))[T.11_25]": "11 a 25 anos",
}
EFFECT_COLORS: dict[str, str] = {
    "Novo": "#134E4A",
    "3 a 10 anos": "#0D9488",
    "11 a 25 anos": "#5EEAD4",
}

# Recent apartments in the 30 busiest neighbourhoods keep the design
# matrix small enough for nine quantile fits.
quantile_data = hedonic_data.loc[
    (hedonic_data["property_type"] == "apartment")
    & hedonic_data["year"].between(2023, 2025)
]
quantile_data = quantile_data.loc[
    quantile_data["neighbourhood"].isin(
        quantile_data["neighbourhood"].value_counts().head(30).index
    )
]

quantile_model = smf.quantreg(
    "log_price ~ log_area"
    " + C(building_age_band, Treatment('26_plus'))"
    " + C(neighbourhood) + C(year)",
    data=quantile_data,
)

quantile_rows: list[dict] = []
for quantile in QUANTILES:
    fit = quantile_model.fit(q=quantile)
    intervals = fit.conf_int()
    for term in ["log_area", *EFFECTS]:
        quantile_rows.append(
            {
                "quantile": quantile,
                "term": term,
                "estimate": fit.params[term],
                "lower": intervals.loc[term, 0],
                "upper": intervals.loc[term, 1],
            }
        )
quantile_effects = pd.DataFrame(quantile_rows)

# Age-band coefficients are log differences; show them as percentages.
premiums = quantile_effects.loc[quantile_effects["term"].isin(EFFECTS)].copy()
premiums["label"] = premiums["term"].map(EFFECTS)
premiums[["estimate", "lower", "upper"]] = (
    np.exp(premiums[["estimate", "lower", "upper"]]) - 1
)
elasticity = quantile_effects.loc[quantile_effects["term"] == "log_area"]

fig_quantile = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.1,
    subplot_titles=[
        "Prêmio sobre imóveis de 26 anos ou mais",
        "Elasticidade do preço por m² à área",
    ],
)

for label, color in EFFECT_COLORS.items():
    subset = premiums.loc[premiums["label"] == label]
    red, green, blue = (int(color[i : i + 2], 16) for i in (1, 3, 5))
    fig_quantile.add_trace(
        go.Scatter(
            x=pd.concat([subset["quantile"], subset["quantile"][::-1]]),
            y=pd.concat([subset["upper"], subset["lower"][::-1]]),
            mode="lines",
            fill="toself",
            fillcolor=f"rgba({red},{green},{blue},0.18)",
            line={"width": 0},
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=1,
    )
    fig_quantile.add_trace(
        go.Scatter(
            x=subset["quantile"],
            y=subset["estimate"],
            mode="lines+markers",
            name=label,
            line={"color": color, "width": 2.5},
            marker={"size": 7, "line": {"color": "white", "width": 1.5}},
            hovertemplate=f"{label}: %{{y:+.1%}}<extra></extra>",
        ),
        row=1,
        col=1,
    )

fig_quantile.add_trace(
    go.Scatter(
        x=pd.concat([elasticity["quantile"], elasticity["quantile"][::-1]]),
        y=pd.concat([elasticity["upper"], elasticity["lower"][::-1]]),
        mode="lines",
        fill="toself",
        fillcolor="rgba(100,116,139,0.18)",
        line={"width": 0},
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig_quantile.add_trace(
    go.Scatter(
        x=elasticity["quantile"],
        y=elasticity["estimate"],
        mode="lines+markers",
        line={"color": INK, "width": 2.5},
        marker={"size": 7, "line": {"color": "white", "width": 1.5}},
        hovertemplate="Elasticidade: %{y:.3f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig_quantile.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: Os efeitos mudam ao longo da distribuição de preços",
        "Regressão quantílica do log do valor real por m² - Apartamentos, "
        "30 bairros com mais transações (2023–2025) - Faixas: IC 95%",
    ),
    height=600,
    hovermode="x unified",
    # Below the panels, so it cannot collide with the panel titles.
    legend=LEGEND | {"y": -0.2, "yanchor": "top"},
)
fig_quantile.update_layout(margin={"b": 100})
fig_quantile.update_xaxes(
    title_text="Quantil do preço por m²",
    tickformat=".0%",
    dtick=0.1,
    showgrid=False,
)
fig_quantile.update_yaxes(gridcolor=GRID, zerolinecolor="#94A3B8")
fig_quantile.update_yaxes(tickformat="+.0%", col=1)

fig_quantile.show()

In [ ]:
import statsmodels.api as sm

BUNCH_BIN: int = 10_000
BUNCH_STEP: int = 50_000
POLYNOMIAL_DEGREE: int = 7

bunching_data = residential.loc[
    residential["year"].between(2023, 2025)
    & residential["value_declared_brl"].between(95_000, 1_005_000)
]

# Bins of R$ 10 mil centred on multiples of R$ 10 mil.
bunch_centres = (
    (bunching_data["value_declared_brl"] / BUNCH_BIN).round() * BUNCH_BIN
).astype(int)
bunching = (
    bunch_centres.value_counts()
    .sort_index()
    .rename_axis("value")
    .reset_index(name="observed")
)
bunching["is_round"] = bunching["value"] % BUNCH_STEP == 0

# Counterfactual: a smooth Poisson polynomial fitted only to the bins
# away from the round numbers, then rescaled to keep the total count.
scaled_value = (
    bunching["value"] - bunching["value"].mean()
) / bunching["value"].std()
design = sm.add_constant(
    np.vander(scaled_value, POLYNOMIAL_DEGREE + 1, increasing=True)[:, 1:]
)
counterfactual_model = sm.GLM(
    bunching.loc[~bunching["is_round"], "observed"],
    design[~bunching["is_round"].to_numpy()],
    family=sm.families.Poisson(),
).fit()
bunching["counterfactual"] = counterfactual_model.predict(design)
bunching["counterfactual"] *= (
    bunching["observed"].sum() / bunching["counterfactual"].sum()
)
bunching["excess"] = bunching["observed"] / bunching["counterfactual"] - 1
round_bins = bunching.loc[bunching["is_round"]]

fig_bunching = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.62, 0.38],
    vertical_spacing=0.08,
)

for is_round, label, color in [
    (False, "Demais faixas", NEUTRAL),
    (True, "Múltiplos de R$ 50 mil", TEAL),
]:
    subset = bunching.loc[bunching["is_round"] == is_round]
    fig_bunching.add_trace(
        go.Bar(
            x=subset["value"],
            y=subset["observed"],
            width=BUNCH_BIN * 0.85,
            name=label,
            marker={"color": color, "line": {"width": 0}},
            hovertemplate="Observado: %{y:,}<extra></extra>",
        ),
        row=1,
        col=1,
    )

fig_bunching.add_trace(
    go.Scatter(
        x=bunching["value"],
        y=bunching["counterfactual"],
        mode="lines",
        name="Contrafactual suave",
        line={"color": INK, "width": 2},
        hovertemplate="Contrafactual: %{y:,.0f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Lollipops: how far each round number sits above its counterfactual.
for value, excess in zip(round_bins["value"], round_bins["excess"]):
    fig_bunching.add_trace(
        go.Scatter(
            x=[value, value],
            y=[0, excess],
            mode="lines",
            line={"color": TEAL, "width": 2},
            hoverinfo="skip",
            showlegend=False,
        ),
        row=2,
        col=1,
    )
fig_bunching.add_trace(
    go.Scatter(
        x=round_bins["value"],
        y=round_bins["excess"],
        mode="markers",
        marker={
            "color": TEAL,
            "size": 10,
            "line": {"color": "white", "width": 1.5},
        },
        hovertemplate="Excesso: %{y:+.0%}<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig_bunching.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        "Belo Horizonte: Quanta massa os números redondos atraem",
        f"Apartamentos e casas (2023–2025), faixas de 10 mil reais - "
        f"Contrafactual: polinômio de Poisson de grau {POLYNOMIAL_DEGREE} "
        f"ajustado fora dos múltiplos de 50 mil",
    ),
    height=720,
    barmode="overlay",
    hovermode="x unified",
    legend=LEGEND,
)
fig_bunching.update_xaxes(showgrid=False)
fig_bunching.update_xaxes(
    title_text="Valor declarado",
    tickprefix="R$ ",
    tickformat=",.0f",
    dtick=100_000,
    row=2,
    col=1,
)
fig_bunching.update_yaxes(gridcolor=GRID, zeroline=False)
fig_bunching.update_yaxes(
    title_text="Transações", tickformat=",.0f", row=1, col=1
)
fig_bunching.update_yaxes(
    title_text="Excesso sobre o contrafactual",
    tickformat="+.0%",
    zeroline=True,
    zerolinecolor="#94A3B8",
    row=2,
    col=1,
)

fig_bunching.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

N_CLUSTERS: int = 4
MIN_SALES_PER_YEAR: int = 30
CLUSTER_COLORS: list[str] = ["#0F766E", "#2563EB", "#D97706", "#7C3AED"]

yearly = (
    residential.loc[
        (residential["property_type"] == "apartment")
        & residential["year"].between(2016, 2025)
        & residential["price_per_m2_brl_real"].notna()
    ]
    .groupby(["neighbourhood", "year"])["price_per_m2_brl_real"]
    .agg(median="median", transactions="size")
    .reset_index()
)
yearly = yearly.loc[yearly["transactions"] >= MIN_SALES_PER_YEAR]

# Only neighbourhoods with a reliable median in all ten years, indexed
# to 2016 so the clusters describe shape and not price level.
trajectories = yearly.pivot(
    index="neighbourhood", columns="year", values="median"
).dropna()
trajectories = 100 * trajectories.div(trajectories[2016], axis=0)
trajectories.index = trajectories.index.str.title()

silhouettes: dict[int, float] = {}
for k in range(2, 7):
    labels = KMeans(
        n_clusters=k, n_init=20, random_state=RANDOM_STATE
    ).fit_predict(trajectories)
    silhouettes[k] = silhouette_score(trajectories, labels)
print(
    "Silhueta por número de grupos:",
    {k: round(score, 3) for k, score in silhouettes.items()},
)

kmeans = KMeans(
    n_clusters=N_CLUSTERS, n_init=20, random_state=RANDOM_STATE
).fit(trajectories)

# Order the clusters from the strongest to the weakest 2025 outcome.
cluster_order = np.argsort(-kmeans.cluster_centers_[:, -1])
years = trajectories.columns.to_numpy()

fig_clusters = make_subplots(
    rows=1,
    cols=N_CLUSTERS,
    shared_yaxes=True,
    horizontal_spacing=0.03,
    subplot_titles=[
        f"Grupo {position} - {(kmeans.labels_ == cluster).sum()} bairros"
        for position, cluster in enumerate(cluster_order, start=1)
    ],
)

for position, cluster in enumerate(cluster_order, start=1):
    members = trajectories.loc[kmeans.labels_ == cluster]
    for neighbourhood, series in members.iterrows():
        fig_clusters.add_trace(
            go.Scatter(
                x=years,
                y=series,
                mode="lines",
                line={"color": NEUTRAL, "width": 1},
                name=neighbourhood,
                hovertemplate=f"{neighbourhood}: %{{y:.0f}}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=position,
        )
    fig_clusters.add_trace(
        go.Scatter(
            x=years,
            y=kmeans.cluster_centers_[cluster],
            mode="lines",
            line={"color": CLUSTER_COLORS[position - 1], "width": 3.5},
            hovertemplate="Centro do grupo: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=position,
    )

fig_clusters.add_hline(
    y=100, line={"color": "#94A3B8", "width": 1, "dash": "dot"}
)

fig_clusters.update_layout(
    **BASE_LAYOUT,
    title=make_title(
        f"Belo Horizonte: Trajetórias de bairro em {N_CLUSTERS} grupos",
        f"Apartamentos - Mediana anual real por m², 2016 = 100 - "
        f"k-médias sobre {len(trajectories)} bairros - Silhueta de "
        f"{silhouettes[N_CLUSTERS]:.2f}: os grupos descrevem um contínuo, "
        f"sem separação nítida".replace(".", ","),
    ),
    height=560,
)
fig_clusters.update_xaxes(
    tickvals=[2017, 2020, 2023], showgrid=False
)
fig_clusters.update_yaxes(gridcolor=GRID, zeroline=False)
fig_clusters.update_yaxes(title_text="Índice (2016 = 100)", col=1)

fig_clusters.show()

Silhueta por número de grupos: {2: np.float64(0.352), 3: np.float64(0.245), 4: np.float64(0.253), 5: np.float64(0.219), 6: np.float64(0.202)}
